# Agent Framework Workflow Evaluation with Response ID Tracking

This notebook demonstrates:
1. Setting up a multi-agent workflow similar to workflow_evaluation.py
2. Capturing response IDs from specific agents during execution
3. Passing response IDs and tool calls to Azure AI Evaluation evaluators
4. Comprehensive evaluation pipeline integration

## Prerequisites
- Azure AI Foundry project configured (az login + environment variables)
- Agent Framework and Azure AI Evaluation packages installed
- Environment variables for Azure AI Foundry set correctly

In [2]:
# Import required libraries
import asyncio
import os
from typing import Dict, List, Any

# Import workflow function and utilities
from workflow_evaluation import run_workflow_with_response_tracking
from tool_utils import extract_tool_definitions_by_agent, format_tool_calls_for_converter

# Evaluation imports
from azure.ai.evaluation import ToolCallAccuracyEvaluator

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

print("Libraries imported successfully")

Libraries imported successfully


## Step 1: Run Workflow and Capture Response Data
Execute the workflow with a sample query and capture all response IDs and tool calls.

In [3]:
# Test query for demonstration - uses multiple agent capabilities
test_query = "Find Microsoft's stock ticker, get its daily performance data, and check the current weather in Seattle"

# Run the workflow and capture response data
workflow_results = await run_workflow_with_response_tracking(test_query)

print("WORKFLOW EXECUTION SUMMARY")
print("=" * 50)
print(f"Query: {workflow_results['query']}")
print(f"Total Interactions: {len(workflow_results['interaction_sequence'])}")
print(f"Tool Calls: {len([i for i in workflow_results['interaction_sequence'] if i['type'] == 'tool_call'])}")
print(f"Tool Results: {len([i for i in workflow_results['interaction_sequence'] if i['type'] == 'tool_result'])}")
print(f"Agent Responses: {len([i for i in workflow_results['interaction_sequence'] if i['type'] == 'agent_response'])}")

print("\nLATEST RESPONSE IDs BY AGENT:")
for agent, response_id in workflow_results['agent_response_ids'].items():
    print(f"  {agent}: {response_id}")

print("\nALL RESPONSE IDs BY AGENT:")
for agent, response_ids in workflow_results['all_agent_response_ids'].items():
    print(f"  {agent}: {len(response_ids)} response IDs")
    for i, response_id in enumerate(response_ids, 1):
        print(f"    {i}. {response_id}")

print("\nLATEST MESSAGE IDs BY AGENT:")
for agent, message_id in workflow_results['agent_message_ids'].items():
    print(f"  {agent}: {message_id}")

print("\nALL MESSAGE IDs BY AGENT:")
for agent, message_ids in workflow_results['all_agent_message_ids'].items():
    print(f"  {agent}: {len(message_ids)} message IDs")
    for i, message_id in enumerate(message_ids, 1):
        print(f"    {i}. {message_id}")

print("\nCOMPLETE INTERACTION SEQUENCE:")
for interaction in workflow_results['interaction_sequence']:
    if interaction['type'] == 'tool_call':
        args_str = ", ".join([f"{k}='{v}'" if isinstance(v, str) else f"{k}={v}" 
                             for k, v in interaction['args'].items()])
        print(f"  {interaction['step']}. [TOOL CALL] {interaction['agent']}: {interaction['function']}({args_str})")
    elif interaction['type'] == 'tool_result':
        result_preview = str()
        print(f"  {interaction['step']}. [TOOL RESULT] {interaction['agent']}: {interaction['result']}")
    elif interaction['type'] == 'agent_response':
        print(f"  {interaction['step']}. [AGENT RESPONSE] {interaction['agent']}: {interaction['message']}")
    elif interaction['type'] == 'final_output':
        print(f"  {interaction['step']}. [FINAL OUTPUT] {interaction['agent']}: {interaction['message']}")


Starting Workflow: 'Find Microsoft's stock ticker, get its daily performance data, and check the current weather in Seattle'
Agent 'financial_tools_assistant' Response ID: run_q9fV56mSMnoFrl4zueBExS6f
Agent 'financial_tools_assistant' Message ID: run_q9fV56mSMnoFrl4zueBExS6f
Agent 'weather_tools_assistant' Response ID: run_WGAECBwKUojK9mebemBLtKFZ
Agent 'weather_tools_assistant' Message ID: run_WGAECBwKUojK9mebemBLtKFZ
Agent 'financial_tools_assistant' Response ID: run_q9fV56mSMnoFrl4zueBExS6f
Agent 'financial_tools_assistant' Message ID: run_q9fV56mSMnoFrl4zueBExS6f
Agent 'weather_tools_assistant' Response ID: run_WGAECBwKUojK9mebemBLtKFZ
Agent 'weather_tools_assistant' Message ID: run_WGAECBwKUojK9mebemBLtKFZ


[2025-10-26 13:03:06 - c:\Users\selshafey\AppData\Local\Programs\Python\Python312\Lib\site-packages\agent_framework\_clients.py:704 - WARNING] When conversation_id is set, store must be True for service-managed threads. Automatically setting store=True.
[2025-10-26 13:03:06 - c:\Users\selshafey\AppData\Local\Programs\Python\Python312\Lib\site-packages\agent_framework\_clients.py:704 - WARNING] When conversation_id is set, store must be True for service-managed threads. Automatically setting store=True.
[2025-10-26 13:03:06 - c:\Users\selshafey\AppData\Local\Programs\Python\Python312\Lib\site-packages\agent_framework\_clients.py:704 - WARNING] When conversation_id is set, store must be True for service-managed threads. Automatically setting store=True.


Agent 'weather_tools_assistant' Response ID: run_WGAECBwKUojK9mebemBLtKFZ
Agent 'weather_tools_assistant' Message ID: run_WGAECBwKUojK9mebemBLtKFZ
1. [TOOL CALL] weather_tools_assistant: get_current_weather(location='Seattle')
2. [TOOL CALL] weather_tools_assistant: get_historical_weather(location='Microsoft')
Agent 'financial_tools_assistant' Response ID: run_q9fV56mSMnoFrl4zueBExS6f
Agent 'financial_tools_assistant' Message ID: run_q9fV56mSMnoFrl4zueBExS6f
3. [TOOL CALL] financial_tools_assistant: find_stock_ticker(company_name='Microsoft')
4. [TOOL CALL] financial_tools_assistant: get_daily_time_series(ticker='MSFT', output_size='compact')
Agent 'weather_tools_assistant' Response ID: run_WGAECBwKUojK9mebemBLtKFZ
Agent 'weather_tools_assistant' Message ID: run_WGAECBwKUojK9mebemBLtKFZ
5. [TOOL RESULT] weather_tools_assistant: {"location": "Seattle", "temperature": "20\u00b0C (Mock) - Sunny", "conditions": "Clear skies"}
6. [TOOL RESULT] weather_tools_assistant: None
Agent 'financial_

[2025-10-26 13:03:06 - c:\Users\selshafey\AppData\Local\Programs\Python\Python312\Lib\site-packages\agent_framework\_clients.py:704 - WARNING] When conversation_id is set, store must be True for service-managed threads. Automatically setting store=True.


Agent 'general_tools_assistant' Response ID: run_8y4Vv7xxRfZOInZe7HTdlxrd
Agent 'general_tools_assistant' Message ID: run_8y4Vv7xxRfZOInZe7HTdlxrd
9. [TOOL CALL] general_tools_assistant: google_search(query='Microsoft stock ticker')
10. [TOOL CALL] general_tools_assistant: google_search(query='current weather in Seattle')
Agent 'general_tools_assistant' Response ID: run_8y4Vv7xxRfZOInZe7HTdlxrd
Agent 'general_tools_assistant' Message ID: run_8y4Vv7xxRfZOInZe7HTdlxrd
11. [TOOL RESULT] general_tools_assistant: {"search_query": "Microsoft stock ticker", "snippet": "A relevant web page snippet was found.", "source": "https://mock.search.engine/result"}
12. [TOOL RESULT] general_tools_assistant: {"search_query": "current weather in Seattle", "snippet": "A relevant web page snippet was found.", "source": "https://mock.search.engine/result"}
Agent 'weather_tools_assistant' Response ID: run_WGAECBwKUojK9mebemBLtKFZ
Agent 'weather_tools_assistant' Message ID: run_WGAECBwKUojK9mebemBLtKFZ
Agent 

[2025-10-26 13:03:10 - c:\Users\selshafey\AppData\Local\Programs\Python\Python312\Lib\site-packages\agent_framework\_clients.py:704 - WARNING] When conversation_id is set, store must be True for service-managed threads. Automatically setting store=True.


Agent 'general_tools_assistant' Response ID: run_8y4Vv7xxRfZOInZe7HTdlxrd
Agent 'general_tools_assistant' Message ID: run_8y4Vv7xxRfZOInZe7HTdlxrd
16. [TOOL RESULT] general_tools_assistant: {"search_query": "Microsoft daily stock performance data", "snippet": "A relevant web page snippet was found.", "source": "https://mock.search.engine/result"}
Agent 'general_tools_assistant' Response ID: run_8y4Vv7xxRfZOInZe7HTdlxrd
Agent 'general_tools_assistant' Message ID: run_8y4Vv7xxRfZOInZe7HTdlxrd
Agent 'general_tools_assistant' Response ID: run_8y4Vv7xxRfZOInZe7HTdlxrd
Agent 'general_tools_assistant' Message ID: run_8y4Vv7xxRfZOInZe7HTdlxrd
Agent 'general_tools_assistant' Response ID: run_8y4Vv7xxRfZOInZe7HTdlxrd
Agent 'general_tools_assistant' Message ID: run_8y4Vv7xxRfZOInZe7HTdlxrd
Agent 'general_tools_assistant' Response ID: run_8y4Vv7xxRfZOInZe7HTdlxrd
Agent 'general_tools_assistant' Message ID: run_8y4Vv7xxRfZOInZe7HTdlxrd
Agent 'general_tools_assistant' Response ID: run_8y4Vv7xxRfZOIn

In [4]:
# Display the enhanced workflow results structure with all response IDs
print("ENHANCED WORKFLOW RESULTS STRUCTURE:")
print("=" * 45)
print(f"Keys: {list(workflow_results.keys())}")
print(f"\nInteraction sequence length: {len(workflow_results['interaction_sequence'])}")
print(f"Tool calls (backwards compatibility): {len(workflow_results['sequential_tool_calls'])}")

print(f"\nResponse ID Tracking Summary:")
total_response_ids = sum(len(ids) for ids in workflow_results['all_agent_response_ids'].values())
total_message_ids = sum(len(ids) for ids in workflow_results['all_agent_message_ids'].values())
print(f"  Total Response IDs captured: {total_response_ids}")
print(f"  Total Message IDs captured: {total_message_ids}")
print(f"  Agents tracked: {len(workflow_results['all_agent_response_ids'])}")

print(f"\nDetailed Response ID Breakdown:")
for agent, response_ids in workflow_results['all_agent_response_ids'].items():
    print(f"  {agent}: {len(response_ids)} response IDs")

print(f"\nDetailed Message ID Breakdown:")
for agent, message_ids in workflow_results['all_agent_message_ids'].items():
    print(f"  {agent}: {len(message_ids)} message IDs")

# Show comparison between latest and all IDs
for agent in workflow_results['all_agent_response_ids'].keys():
    all_response_ids = workflow_results['all_agent_response_ids'][agent]
    print(f"  {agent}:")
    print(f"    All Response IDs: {all_response_ids}")


ENHANCED WORKFLOW RESULTS STRUCTURE:
Keys: ['interaction_sequence', 'sequential_tool_calls', 'agent_response_ids', 'agent_message_ids', 'all_agent_response_ids', 'all_agent_message_ids', 'query']

Interaction sequence length: 18
Tool calls (backwards compatibility): 7

Response ID Tracking Summary:
  Total Response IDs captured: 3
  Total Message IDs captured: 3
  Agents tracked: 3

Detailed Response ID Breakdown:
  financial_tools_assistant: 1 response IDs
  weather_tools_assistant: 1 response IDs
  general_tools_assistant: 1 response IDs

Detailed Message ID Breakdown:
  financial_tools_assistant: 1 message IDs
  weather_tools_assistant: 1 message IDs
  general_tools_assistant: 1 message IDs
  financial_tools_assistant:
    All Response IDs: {'run_q9fV56mSMnoFrl4zueBExS6f'}
  weather_tools_assistant:
    All Response IDs: {'run_WGAECBwKUojK9mebemBLtKFZ'}
  general_tools_assistant:
    All Response IDs: {'run_8y4Vv7xxRfZOInZe7HTdlxrd'}


## Step 2: Prepare Data for Evaluation
Format the captured tool calls and prepare tool definitions for the evaluator.

In [5]:
# Get tool definitions for evaluation
all_tool_definitions = extract_tool_definitions_by_agent()

# Format tool calls for converter
formatted_tool_calls = format_tool_calls_for_converter(workflow_results['sequential_tool_calls'])

# Filter for financial tools assistant (as example)
financial_tool_calls = [
    tool_call for tool_call in formatted_tool_calls 
    if tool_call['agent'] == 'financial_tools_assistant'
]

# Get financial tool definitions
financial_tool_definitions = all_tool_definitions.get('financial_tools_assistant', [])

print("EVALUATION DATA PREPARATION")
print("=" * 40)
print(f"Total Tool Calls: {len(formatted_tool_calls)}")
print(f"Financial Tool Calls: {len(financial_tool_calls)}")
print(f"Financial Tool Definitions: {len(financial_tool_definitions)}")

print("\nFORMATTED TOOL CALLS:")
for i, call in enumerate(formatted_tool_calls, 1):
    response_id = next((tc['response_id'] for tc in workflow_results['sequential_tool_calls'] 
                       if tc['call_id'] == call['call_id']), 'N/A')
    print(f"{i}. Agent: {call['agent']}")
    print(f"   Function: {call['name']}")
    print(f"   Arguments: {call['arguments']}")
    print(f"   Call ID: {call['call_id']}")
    print(f"   Response ID: {response_id}")
    print()

print("FINANCIAL TOOL DEFINITIONS:")
for i, tool_def in enumerate(financial_tool_definitions, 1):
    print(f"{i}. {tool_def['name']}: {tool_def['description']}")

EVALUATION DATA PREPARATION
Total Tool Calls: 7
Financial Tool Calls: 2
Financial Tool Definitions: 3

FORMATTED TOOL CALLS:
1. Agent: weather_tools_assistant
   Function: get_current_weather
   Arguments: {'location': 'Seattle'}
   Call ID: ["run_WGAECBwKUojK9mebemBLtKFZ", "call_BE1wsxMiCszGl7nCFJUkeFTe"]
   Response ID: run_WGAECBwKUojK9mebemBLtKFZ

2. Agent: weather_tools_assistant
   Function: get_historical_weather
   Arguments: {'location': 'Microsoft'}
   Call ID: ["run_WGAECBwKUojK9mebemBLtKFZ", "call_kJuurM2fW1er7flkjGQLq6zX"]
   Response ID: run_WGAECBwKUojK9mebemBLtKFZ

3. Agent: financial_tools_assistant
   Function: find_stock_ticker
   Arguments: {'company_name': 'Microsoft'}
   Call ID: ["run_q9fV56mSMnoFrl4zueBExS6f", "call_8fRrLNOeWnTtrUc2h5h51IzN"]
   Response ID: run_q9fV56mSMnoFrl4zueBExS6f

4. Agent: financial_tools_assistant
   Function: get_daily_time_series
   Arguments: {'ticker': 'MSFT', 'output_size': 'compact'}
   Call ID: ["run_q9fV56mSMnoFrl4zueBExS6f", "c

## Step 3: Run Azure AI Evaluation with Response IDs
Execute the ToolCallAccuracyEvaluator with our captured response IDs and tool calls.

In [8]:
# Debug the final output issue
print("DEBUGGING FINAL OUTPUT ISSUE")
print("=" * 40)

# Check the workflow results structure
if 'workflow_results' in locals() and workflow_results:
    print("Workflow results keys:", list(workflow_results.keys()))
    
    # Look at the interaction sequence around the final steps
    sequence = workflow_results.get('interaction_sequence', [])
    print(f"\nTotal interactions: {len(sequence)}")
    
    # Show the last few interactions
    print("\nLast 5 interactions:")
    for item in sequence[-5:]:
        if item['type'] == 'agent_response':
            print(f"  {item['step']}. [AGENT RESPONSE] {item['agent']}: {item.get('message', 'NO MESSAGE')[:100]}...")
        elif item['type'] == 'final_output':
            print(f"  {item['step']}. [FINAL OUTPUT] {item['agent']}: '{item.get('message', 'NO MESSAGE')}'")
        elif item['type'] == 'tool_call':
            print(f"  {item['step']}. [TOOL CALL] {item['agent']}: {item['function']}")
        elif item['type'] == 'tool_result':
            print(f"  {item['step']}. [TOOL RESULT] {item['agent']}: {str(item['result'])[:50]}...")
    
    # Check if final output exists and what it contains
    final_outputs = [item for item in sequence if item['type'] == 'final_output']
    print(f"\nFinal outputs found: {len(final_outputs)}")
    for output in final_outputs:
        print(f"  Final output message: '{output.get('message', 'EMPTY')}'")
        print(f"  Message type: {type(output.get('message'))}")
        print(f"  Message length: {len(str(output.get('message', '')))}")
        
else:
    print("No workflow_results variable found - run the workflow first")

print("\nPOSSIBLE CAUSES:")
print("1. Research lead agent failed to generate a response")
print("2. Final response message is empty or None")
print("3. Error in fan_in_handle method during message processing")
print("4. Agent configuration issue (missing deployment/credentials)")
print("5. Context or message format issue preventing proper response generation")

DEBUGGING FINAL OUTPUT ISSUE
Workflow results keys: ['interaction_sequence', 'sequential_tool_calls', 'agent_response_ids', 'agent_message_ids', 'all_agent_response_ids', 'all_agent_message_ids', 'query']

Total interactions: 18

Last 5 interactions:
  14. [TOOL RESULT] weather_tools_assistant: {"location": "Seattle", "historical_data": {"date"...
  15. [AGENT RESPONSE] financial_tools_assistant: Microsoft's official stock ticker is **MOCK** (on NASDAQ). 

Its daily stock performance data is as ...
  16. [AGENT RESPONSE] general_tools_assistant: Here are the results of the tasks:

1. **Microsoft's Stock Ticker**: Microsoft's stock ticker symbol...
  17. [AGENT RESPONSE] weather_tools_assistant: I can provide the following:

1. **Current weather in Seattle**: It is 20°C, sunny with clear skies....
  18. [FINAL OUTPUT] research_lead: ''

Final outputs found: 1
  Final output message: ''
  Message type: <class 'str'>
  Message length: 0

POSSIBLE CAUSES:
1. Research lead agent failed to g

In [9]:
# Test the updated workflow with enhanced final output handling
print("Testing updated workflow with enhanced final output...")

# Reload the module to get the latest changes
import importlib
import sys
if 'workflow_evaluation' in sys.modules:
    importlib.reload(sys.modules['workflow_evaluation'])

from workflow_evaluation import run_workflow_with_response_tracking

# Test with the same query that had issues
test_query = "Find Microsoft's stock ticker, get its daily performance data, and check the current weather in Seattle"

try:
    print(f"Running workflow with query: '{test_query}'")
    print("=" * 60)
    
    # Run the updated workflow
    updated_results = await run_workflow_with_response_tracking(test_query)
    
    print("\n✅ Workflow completed successfully!")
    
    # Check the final output specifically
    sequence = updated_results.get('interaction_sequence', [])
    final_outputs = [item for item in sequence if item['type'] == 'final_output']
    
    print(f"\n📋 Final Output Analysis:")
    print(f"   Total interactions: {len(sequence)}")
    print(f"   Final outputs found: {len(final_outputs)}")
    
    for output in final_outputs:
        message = output.get('message', '')
        print(f"   Final output length: {len(message)} characters")
        print(f"   Final output preview: '{message[:200]}{'...' if len(message) > 200 else ''}'")
        
        if message.strip():
            print("   ✅ Final output contains content!")
        else:
            print("   ❌ Final output is still empty")
    
    # Show the last few steps to see the flow
    print(f"\nLast 3 interactions:")
    for item in sequence[-3:]:
        if item['type'] == 'agent_response':
            preview = item.get('message', '')[:100] + "..." if len(item.get('message', '')) > 100 else item.get('message', '')
            print(f"  {item['step']}. [AGENT RESPONSE] {item['agent']}: {preview}")
        elif item['type'] == 'final_output':
            preview = item.get('message', '')[:100] + "..." if len(item.get('message', '')) > 100 else item.get('message', '')
            print(f"  {item['step']}. [FINAL OUTPUT] {item['agent']}: {preview}")
            
except Exception as e:
    print(f"❌ Error during updated workflow execution: {e}")
    import traceback
    traceback.print_exc()

Testing updated workflow with enhanced final output...
Running workflow with query: 'Find Microsoft's stock ticker, get its daily performance data, and check the current weather in Seattle'

Starting Workflow: 'Find Microsoft's stock ticker, get its daily performance data, and check the current weather in Seattle'
Agent 'financial_tools_assistant' Response ID: run_vv9T6NciotPBxHYjDnUiOJh1
Agent 'financial_tools_assistant' Message ID: run_vv9T6NciotPBxHYjDnUiOJh1
Agent 'weather_tools_assistant' Response ID: run_L3ZC9ld2ZwPWbIlAxoH8pd22
Agent 'weather_tools_assistant' Message ID: run_L3ZC9ld2ZwPWbIlAxoH8pd22


[2025-10-26 13:00:29 - c:\Users\selshafey\AppData\Local\Programs\Python\Python312\Lib\site-packages\agent_framework\_clients.py:704 - WARNING] When conversation_id is set, store must be True for service-managed threads. Automatically setting store=True.
[2025-10-26 13:00:29 - c:\Users\selshafey\AppData\Local\Programs\Python\Python312\Lib\site-packages\agent_framework\_clients.py:704 - WARNING] When conversation_id is set, store must be True for service-managed threads. Automatically setting store=True.


Agent 'financial_tools_assistant' Response ID: run_vv9T6NciotPBxHYjDnUiOJh1
Agent 'financial_tools_assistant' Message ID: run_vv9T6NciotPBxHYjDnUiOJh1
1. [TOOL CALL] financial_tools_assistant: find_stock_ticker(company_name='Microsoft')
2. [TOOL CALL] financial_tools_assistant: get_daily_time_series(ticker='MSFT', output_size='compact')
Agent 'financial_tools_assistant' Response ID: run_vv9T6NciotPBxHYjDnUiOJh1
Agent 'financial_tools_assistant' Message ID: run_vv9T6NciotPBxHYjDnUiOJh1
3. [TOOL RESULT] financial_tools_assistant: {"search_name": "microsoft", "ticker_symbol": "MOCK", "exchange": "NASDAQ (Mock)"}
4. [TOOL RESULT] financial_tools_assistant: {"ticker": "MSFT", "output_size": "compact", "last_close": "180.20", "open": "179.80", "volume": "15,500,000", "note": "Mock daily data (Output size: compact)."}
Agent 'general_tools_assistant' Response ID: run_X6MRUD22odwRIuNWbwi5Gz3a
Agent 'general_tools_assistant' Message ID: run_X6MRUD22odwRIuNWbwi5Gz3a
Agent 'weather_tools_assistant

[2025-10-26 13:00:30 - c:\Users\selshafey\AppData\Local\Programs\Python\Python312\Lib\site-packages\agent_framework\_clients.py:704 - WARNING] When conversation_id is set, store must be True for service-managed threads. Automatically setting store=True.


Agent 'general_tools_assistant' Response ID: run_X6MRUD22odwRIuNWbwi5Gz3a
Agent 'general_tools_assistant' Message ID: run_X6MRUD22odwRIuNWbwi5Gz3a
9. [TOOL CALL] general_tools_assistant: google_search(query='Microsoft stock ticker')
10. [TOOL CALL] general_tools_assistant: google_search(query='current weather Seattle')
Agent 'general_tools_assistant' Response ID: run_X6MRUD22odwRIuNWbwi5Gz3a
Agent 'general_tools_assistant' Message ID: run_X6MRUD22odwRIuNWbwi5Gz3a
11. [TOOL RESULT] general_tools_assistant: {"search_query": "Microsoft stock ticker", "snippet": "A relevant web page snippet was found.", "source": "https://mock.search.engine/result"}
12. [TOOL RESULT] general_tools_assistant: {"search_query": "current weather Seattle", "snippet": "A relevant web page snippet was found.", "source": "https://mock.search.engine/result"}
Agent 'financial_tools_assistant' Response ID: run_vv9T6NciotPBxHYjDnUiOJh1
Agent 'financial_tools_assistant' Message ID: run_vv9T6NciotPBxHYjDnUiOJh1
Agent 'f

[2025-10-26 13:00:33 - c:\Users\selshafey\AppData\Local\Programs\Python\Python312\Lib\site-packages\agent_framework\_clients.py:704 - WARNING] When conversation_id is set, store must be True for service-managed threads. Automatically setting store=True.


Agent 'financial_tools_assistant' Response ID: run_vv9T6NciotPBxHYjDnUiOJh1
Agent 'financial_tools_assistant' Message ID: run_vv9T6NciotPBxHYjDnUiOJh1
Agent 'financial_tools_assistant' Response ID: run_vv9T6NciotPBxHYjDnUiOJh1
Agent 'financial_tools_assistant' Message ID: run_vv9T6NciotPBxHYjDnUiOJh1
Agent 'financial_tools_assistant' Response ID: run_vv9T6NciotPBxHYjDnUiOJh1
Agent 'financial_tools_assistant' Message ID: run_vv9T6NciotPBxHYjDnUiOJh1
Agent 'financial_tools_assistant' Response ID: run_vv9T6NciotPBxHYjDnUiOJh1
Agent 'financial_tools_assistant' Message ID: run_vv9T6NciotPBxHYjDnUiOJh1
Agent 'financial_tools_assistant' Response ID: run_vv9T6NciotPBxHYjDnUiOJh1
Agent 'financial_tools_assistant' Message ID: run_vv9T6NciotPBxHYjDnUiOJh1
Agent 'financial_tools_assistant' Response ID: run_vv9T6NciotPBxHYjDnUiOJh1
Agent 'financial_tools_assistant' Message ID: run_vv9T6NciotPBxHYjDnUiOJh1
Agent 'financial_tools_assistant' Response ID: run_vv9T6NciotPBxHYjDnUiOJh1
Agent 'financial_t

In [10]:
# Quick check of the final output from the latest test
if 'updated_results' in locals() and updated_results:
    print("FINAL OUTPUT ANALYSIS")
    print("=" * 30)
    
    sequence = updated_results.get('interaction_sequence', [])
    final_outputs = [item for item in sequence if item['type'] == 'final_output']
    
    print(f"Total interactions: {len(sequence)}")
    print(f"Final outputs found: {len(final_outputs)}")
    
    for i, output in enumerate(final_outputs, 1):
        message = output.get('message', '')
        print(f"\nFinal Output {i}:")
        print(f"  Length: {len(message)} characters")
        print(f"  Empty: {'Yes' if not message.strip() else 'No'}")
        if message.strip():
            # Show first 300 characters
            print(f"  Preview: {message[:300]}{'...' if len(message) > 300 else ''}")
        else:
            print(f"  Content: '{message}'")
    
    # Check the very last interaction
    if sequence:
        last_item = sequence[-1]
        print(f"\nLast interaction:")
        print(f"  Step: {last_item['step']}")
        print(f"  Type: {last_item['type']}")
        print(f"  Agent: {last_item.get('agent', 'N/A')}")
        if last_item['type'] == 'final_output':
            message = last_item.get('message', '')
            print(f"  Message length: {len(message)}")
            print(f"  Has content: {'Yes' if message.strip() else 'No'}")
            
    print(f"\n🎯 CONCLUSION:")
    if final_outputs and any(output.get('message', '').strip() for output in final_outputs):
        print("✅ Final output issue is RESOLVED - content is now being generated!")
    else:
        print("❌ Final output issue PERSISTS - still empty")
        
else:
    print("No updated_results available - run the test above first")

FINAL OUTPUT ANALYSIS
Total interactions: 18
Final outputs found: 1

Final Output 1:
  Length: 429 characters
  Empty: No
  Preview: Based on the findings:

1. **Microsoft's Stock Ticker and Performance**: Microsoft's stock ticker is **MSFT**. However, no detailed daily performance data was retrieved directly from available tools in this context.

2. **Current Weather in Seattle**: The current weather in Seattle is 20°C, with cle...

Last interaction:
  Step: 18
  Type: final_output
  Agent: research_lead
  Message length: 429
  Has content: Yes

🎯 CONCLUSION:
✅ Final output issue is RESOLVED - content is now being generated!


In [4]:
# Test the updated tool call capture logic
print("Testing updated workflow evaluation with Azure AI Foundry...")

# Clear any previous results
workflow_results = None

# Run the enhanced workflow with Azure AI Foundry
try:
    workflow_results = await run_enhanced_workflow_evaluation()
    print("\n✅ Workflow completed successfully!")
    
    # Display the interaction sequence with tool calls
    if workflow_results and 'interaction_sequence' in workflow_results:
        print(f"\n📋 Complete Interaction Sequence ({len(workflow_results['interaction_sequence'])} steps):")
        for item in workflow_results['interaction_sequence']:
            if item['type'] == 'tool_call':
                args_str = ", ".join([f"{k}='{v}'" if isinstance(v, str) else f"{k}={v}" 
                                    for k, v in item['args'].items()])
                print(f"  {item['step']}. [TOOL CALL] {item['agent']}: {item['function']}({args_str})")
            elif item['type'] == 'tool_result':
                result_preview = str(item['result'])[:100] + "..." if len(str(item['result'])) > 100 else str(item['result'])
                print(f"  {item['step']}. [TOOL RESULT] {item['agent']}: {result_preview}")
            elif item['type'] == 'agent_response':
                response_preview = item['content'][:100] + "..." if len(item['content']) > 100 else item['content']
                print(f"  {item['step']}. [RESPONSE] {item['agent']}: {response_preview}")
    
    # Count tool calls vs tool results
    if workflow_results and 'interaction_sequence' in workflow_results:
        tool_calls = [item for item in workflow_results['interaction_sequence'] if item['type'] == 'tool_call']
        tool_results = [item for item in workflow_results['interaction_sequence'] if item['type'] == 'tool_result']
        agent_responses = [item for item in workflow_results['interaction_sequence'] if item['type'] == 'agent_response']
        
        print(f"\n📊 Sequence Analysis:")
        print(f"   Tool Calls: {len(tool_calls)}")
        print(f"   Tool Results: {len(tool_results)}")
        print(f"   Agent Responses: {len(agent_responses)}")
        
        if len(tool_calls) == 0 and len(tool_results) > 0:
            print("   ⚠️  WARNING: Tool results found but no tool calls captured!")
        elif len(tool_calls) == len(tool_results):
            print("   ✅ Tool calls and results are balanced")
        else:
            print(f"   ⚠️  Mismatch: {len(tool_calls)} calls vs {len(tool_results)} results")
    
except Exception as e:
    print(f"❌ Error during workflow execution: {e}")
    import traceback
    traceback.print_exc()

Testing updated workflow evaluation with Azure AI Foundry...
❌ Error during workflow execution: name 'run_enhanced_workflow_evaluation' is not defined


Traceback (most recent call last):
  File "C:\Users\selshafey\AppData\Local\Temp\ipykernel_24676\381930770.py", line 9, in <module>
    workflow_results = await run_enhanced_workflow_evaluation()
                             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
NameError: name 'run_enhanced_workflow_evaluation' is not defined


In [ ]:
# Test the updated tool call capture - comprehensive test
print("Testing updated workflow evaluation with better error handling...")

# Test with a simple query that should trigger tool calls
test_query = "What is the current weather in Seattle and New York?"

try:
    # Import the updated workflow evaluation function
    import importlib
    import sys
    sys.path.append(r'c:\Users\selshafey\agent-framework\python\samples\getting_started\workflows\evaluation')
    
    # Reload the module to get latest changes
    if 'workflow_evaluation' in sys.modules:
        importlib.reload(sys.modules['workflow_evaluation'])
    
    from workflow_evaluation import run_enhanced_workflow_evaluation
    
    print(f"✅ Successfully imported updated workflow evaluation")
    print(f"🔍 Testing with query: '{test_query}'")
    
    # Run the enhanced workflow
    workflow_results = await run_enhanced_workflow_evaluation(test_query)
    
    print("\n✅ Workflow completed successfully!")
    
    # Analyze the results
    if workflow_results and 'interaction_sequence' in workflow_results:
        sequence = workflow_results['interaction_sequence']
        print(f"\n📋 Complete Interaction Sequence ({len(sequence)} steps):")
        
        tool_calls = []
        tool_results = []
        agent_responses = []
        
        for item in sequence:
            if item['type'] == 'tool_call':
                tool_calls.append(item)
                args_str = ", ".join([f"{k}='{v}'" if isinstance(v, str) else f"{k}={v}" 
                                    for k, v in item['args'].items()])
                print(f"  {item['step']}. [TOOL CALL] {item['agent']}: {item['function']}({args_str})")
                
            elif item['type'] == 'tool_result':
                tool_results.append(item)
                result_preview = str(item['result'])[:100] + "..." if len(str(item['result'])) > 100 else str(item['result'])
                print(f"  {item['step']}. [TOOL RESULT] {item['agent']}: {result_preview}")
                
            elif item['type'] == 'agent_response':
                agent_responses.append(item)
                response_preview = item['content'][:100] + "..." if len(item['content']) > 100 else item['content']
                print(f"  {item['step']}. [RESPONSE] {item['agent']}: {response_preview}")
        
        # Analysis
        print(f"\n📊 Sequence Analysis:")
        print(f"   Tool Calls: {len(tool_calls)}")
        print(f"   Tool Results: {len(tool_results)}")
        print(f"   Agent Responses: {len(agent_responses)}")
        
        if len(tool_calls) == 0 and len(tool_results) > 0:
            print("   ⚠️  WARNING: Tool results found but no tool calls captured!")
            print("   This suggests the Azure AI content type detection needs adjustment")
        elif len(tool_calls) == len(tool_results):
            print("   ✅ Tool calls and results are balanced - capture working correctly!")
        else:
            print(f"   ⚠️  Mismatch: {len(tool_calls)} calls vs {len(tool_results)} results")
            
        # Show response IDs tracking
        if workflow_results.get('response_ids'):
            print(f"\n🆔 Response ID Tracking:")
            print(f"   Total Response IDs captured: {len(workflow_results['response_ids'])}")
            for i, rid in enumerate(workflow_results['response_ids'][:5]):  # Show first 5
                print(f"   {i+1}. {rid}")
            if len(workflow_results['response_ids']) > 5:
                print(f"   ... and {len(workflow_results['response_ids']) - 5} more")
                
    else:
        print("❌ No interaction sequence found in results")
        
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Make sure the workflow_evaluation.py file is accessible")
    
except Exception as e:
    print(f"❌ Error during workflow execution: {e}")
    print(f"Error type: {type(e).__name__}")
    
    # Check if it's an authentication/connection issue
    if "Cannot connect" in str(e) or "authentication" in str(e).lower():
        print("\n💡 This appears to be an authentication or connection issue.")
        print("   Please ensure your Azure AI Foundry or Azure OpenAI credentials are set up correctly.")
        print("   You can check the environment variables in the previous cell.")
    
    import traceback
    traceback.print_exc()